# Shoealls — WearGait-PD 학습 (Google Colab)

**준비:**
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- 왼쪽 자물쇠(🔑) → Secrets → `GITHUB_TOKEN` 추가

**데이터 폴더 ID:** `1JgXO4UdyUyG69ppXfLoSojBJ_MU3fONd`

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Drive 마운트 + 데이터 다운로드
from google.colab import drive
drive.mount('/content/drive')

import subprocess, os

FOLDER_ID = '1JgXO4UdyUyG69ppXfLoSojBJ_MU3fONd'
DATA_DIR  = '/content/weargait_data'

if not os.path.exists(DATA_DIR):
    print('데이터 다운로드 중...')
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    gdown.download_folder(id=FOLDER_ID, output=DATA_DIR, quiet=False, use_cookies=False)
else:
    print('이미 다운로드됨')

all_files = os.listdir(DATA_DIR)
csv_files = [f for f in all_files if f.endswith('.csv') and '_mat' not in f]
print(f'CSV 파일: {len(csv_files)}개 (mat 제외)')

In [ ]:
# 파일명 접두사로 라벨 분류
# WHC* → 건강군(0), WPD* → 파킨슨(3), NLS*(control) → 건강군(0)

def get_label(filename):
    name = filename.upper()
    if name.startswith('WHC'):
        return 0
    if name.startswith('WPD'):
        return 3
    if name.startswith('NLS') and 'CONTROL' in name:
        return 0
    return None  # NLS non-control: 제외

labeled_files = []
for f in csv_files:
    label = get_label(f)
    if label is not None:
        labeled_files.append((os.path.join(DATA_DIR, f), label))

hc = [(p, l) for p, l in labeled_files if l == 0]
pd = [(p, l) for p, l in labeled_files if l == 3]
print(f'건강군(HC): {len(hc)}개, 파킨슨(PD): {len(pd)}개, 총: {len(labeled_files)}개')

In [ ]:
# 레포 클론
from google.colab import userdata
import sys

try:
    token = userdata.get('GITHUB_TOKEN')
except:
    token = input('GitHub Token: ').strip()

if not os.path.exists('/content/Shoealls'):
    os.system(f'git clone https://{token}@github.com/jg-shoealls/Shoealls.git /content/Shoealls 2>&1 | tail -2')

sys.path.insert(0, '/content/Shoealls')
os.chdir('/content/Shoealls')

subprocess.run(['pip', 'install', '-q', 'pandas', 'scipy', 'scikit-learn', 'pyyaml', 'matplotlib'], check=True)
print('준비 완료')

In [ ]:
# 데이터 파싱 (WearGait 어댑터 직접 구현)
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

SEQ_LEN = 128

IMU_COLS = [
    'LowerBack_Acc_X', 'LowerBack_Acc_Y', 'LowerBack_Acc_Z',
    'LowerBack_Gyr_X', 'LowerBack_Gyr_Y', 'LowerBack_Gyr_Z'
]

def parse_csv(path):
    try:
        df = pd.read_csv(path, low_memory=False)
        if not all(c in df.columns for c in IMU_COLS):
            return None
        imu = df[IMU_COLS].values.astype(np.float32)

        # 족저압: L/R 스칼라 → 16x8 더미 그리드
        T = len(df)
        pres_l = df['L Foot Pressure'].values.astype(np.float32) if 'L Foot Pressure' in df.columns else np.zeros(T)
        pres_r = df['R Foot Pressure'].values.astype(np.float32) if 'R Foot Pressure' in df.columns else np.zeros(T)
        pressure = np.zeros((T, 16, 8), dtype=np.float32)
        pressure[:, :8, :] = pres_l[:, None, None] / 255.0
        pressure[:, 8:, :] = pres_r[:, None, None] / 255.0

        # 스켈레톤: 없어서 0으로 채움
        skeleton = np.zeros((T, 17, 3), dtype=np.float32)

        return imu, pressure, skeleton
    except Exception as e:
        return None

def pad_or_crop(arr, seq_len):
    T = len(arr)
    if T >= seq_len:
        return arr[:seq_len]
    pad_width = [(0, seq_len - T)] + [(0, 0)] * (arr.ndim - 1)
    return np.pad(arr, pad_width)

class WearGaitDataset(Dataset):
    def __init__(self, file_label_pairs, seq_len=128):
        self.samples = []
        print('파싱 중...')
        for path, label in file_label_pairs:
            result = parse_csv(path)
            if result is None:
                continue
            imu, pressure, skeleton = result
            self.samples.append((
                pad_or_crop(imu, seq_len),
                pad_or_crop(pressure, seq_len),
                pad_or_crop(skeleton, seq_len),
                label
            ))
        print(f'로드 완료: {len(self.samples)}개 샘플')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        imu, pressure, skeleton, label = self.samples[idx]
        return {
            'imu':      torch.from_numpy(imu),
            'pressure': torch.from_numpy(pressure),
            'skeleton': torch.from_numpy(skeleton),
            'label':    torch.tensor(label, dtype=torch.long)
        }

dataset = WearGaitDataset(labeled_files, seq_len=SEQ_LEN)
print(f'최종 샘플: {len(dataset)}개')

In [ ]:
# 설정 + 모델 초기화
import torch.nn as nn
import yaml

CONFIG_PATH = '/content/Shoealls/configs/default.yaml'
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)
else:
    config = {
        'data': {
            'sequence_length': SEQ_LEN,
            'imu_channels': 6,
            'pressure_grid_size': [16, 8],
            'skeleton_joints': 17,
            'skeleton_dims': 3,
            'num_classes': 4,
        },
        'model': {
            'fusion': {'embed_dim': 128},
            'imu_encoder': {'dropout': 0.1},
            'pressure_encoder': {'dropout': 0.1},
            'skeleton_encoder': {'gcn_channels': [64, 128], 'temporal_kernel': 9, 'dropout': 0.1},
        },
        'training': {'batch_size': 16, 'epochs': 30, 'learning_rate': 1e-3, 'weight_decay': 1e-4}
    }

config['data']['num_classes'] = 4

# 데이터 분할
total = len(dataset)
train_n = int(total * 0.7)
val_n   = int(total * 0.15)
test_n  = total - train_n - val_n
train_ds, val_ds, test_ds = random_split(
    dataset, [train_n, val_n, test_n],
    generator=torch.Generator().manual_seed(42)
)
print(f'학습: {len(train_ds)} / 검증: {len(val_ds)} / 테스트: {len(test_ds)}')

BS = config['training']['batch_size']
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, num_workers=2, pin_memory=True)

# 모델
from src.models.multimodal_gait_net import MultimodalGaitNet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MultimodalGaitNet(config).to(device)
print(f'Device: {device} | 파라미터: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# 학습
import time

EPOCHS   = config['training']['epochs']
SAVE_DIR = '/content/drive/MyDrive/WearGait-PD/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            imu      = batch['imu'].to(device)
            pressure = batch['pressure'].to(device)
            skeleton = batch['skeleton'].to(device)
            labels   = batch['label'].to(device)

            out    = model(imu, pressure, skeleton)
            logits = out if isinstance(out, torch.Tensor) else out['logits']
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)
    return total_loss / total, correct / total

print(f'{'Epoch':>5} | {'TrLoss':>7} | {'TrAcc':>6} | {'VlLoss':>7} | {'VlAcc':>6} | Time')
print('-' * 50)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    flag = ' ★' if vl_acc > best_val_acc else ''
    print(f'{epoch:>5} | {tr_loss:>7.4f} | {tr_acc:>6.4f} | {vl_loss:>7.4f} | {vl_acc:>6.4f} | {time.time()-t0:.1f}s{flag}')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'config': config,
            'val_accuracy': best_val_acc,
            'model_type': 'basic'
        }, os.path.join(SAVE_DIR, 'best_model.pt'))

print(f'\n완료! 최고 Val Acc: {best_val_acc:.4f}')

In [ ]:
# 테스트 평가
checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pt'))
model.load_state_dict(checkpoint['model_state_dict'])
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f'테스트 정확도: {test_acc:.4f} ({test_acc*100:.1f}%)')

In [ ]:
# 학습 곡선
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train'); ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
ax2.plot(history['train_acc'], label='Train'); ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curve.png'), dpi=150)
plt.show()

In [ ]:
# 샘플 예측 확인
CLASS_KR = ['정상 보행', '절뚝거림', '운동실조', '파킨슨']
model.eval()
batch = next(iter(test_loader))
with torch.no_grad():
    out    = model(batch['imu'].to(device), batch['pressure'].to(device), batch['skeleton'].to(device))
    logits = out if isinstance(out, torch.Tensor) else out['logits']
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    preds  = probs.argmax(axis=1)
    gts    = batch['label'].numpy()

print(f'{'#':>3} | {'실제':^8} | {'예측':^8} | 신뢰도 | 정오')
print('-' * 42)
for i in range(min(10, len(gts))):
    ok = '✓' if preds[i] == gts[i] else '✗'
    print(f'{i+1:>3} | {CLASS_KR[gts[i]]:^8} | {CLASS_KR[preds[i]]:^8} | {probs[i][preds[i]]:.1%} | {ok}')